Set up

In [42]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import fnmatch
import re
import pingouin as pg

from scipy.stats import spearmanr
from transformers import CLIPProcessor, CLIPModel

import matplotlib.pyplot as plt

# Define paths 
data_folder = "../../Drawings2/exp_gazeCon"
embeddings_folder = "../../dnn_features/clip/exp_gazeCon" 
if not os.path.exists(embeddings_folder):
    os.makedirs(embeddings_folder)
cates = ["bat", "kit"] 

category_names = {
    "bat": "bathroom",
    "kit": "kitchen",
}


Get Embeddings

In [36]:
# Define category ('bat' or 'kit')
model = "clip"

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# -------- LOAD IMAGES --------
def load_images(folder):
    exts = (".png") 
    files = [os.path.join(folder, f) for f in os.listdir(folder) 
             if (f.lower().endswith(exts) and 'raw' not in f)]
    return sorted(files)

#----------LOAD CAPTIONS----------
def load_captions(folder, num_repeats=1):
    caption_file = os.path.join(folder, "objects_all.csv")
    if os.path.exists(caption_file):
        # read text file into pandas dataframe
        captions_df = pd.read_csv(caption_file, sep=",", header=0)

        # Repeat each row X times
        captions_df = captions_df.loc[captions_df.index.repeat(num_repeats)].reset_index(drop=True)
        return captions_df
    else:
        print("Caption file not found.")
    return []

# -------- EXTRACT EMBEDDINGS --------
def get_embedding(img, caption, model, processor, device):

    inputs = processor(images=img, text=[caption], return_tensors="pt", padding=True).to(device)
    
    with torch.no_grad():

        inputs = processor(images=img, text=[caption], return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            outputs = clip_model(**inputs)

        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

        # Safety check
        assert isinstance(image_embeds, torch.Tensor)
        assert isinstance(text_embeds, torch.Tensor)

        # Normalize
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True) + 1e-8
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True) + 1e-8

        # Joint embedding
        joint_embedding = (image_embeds + text_embeds) / 2



    return image_embeds, text_embeds, joint_embedding

def loop_images(data_folder, embeddings_folder, model, processor, device, num_repeats=1):

    # Load images and captions
    image_paths = load_images(data_folder)
    print(f"Found {len(image_paths)} drawings.")

    captions = load_captions(data_folder, num_repeats=num_repeats)
    print(f"Found {len(captions)} captions.")

    all_joined_embeddings = []
    all_text_embeddings = []
    all_image_embeddings = []
    filename_all = []

    for iDrw in tqdm(range(0, captions.shape[0]), desc="Extracting embeddings"):

        # find corresponding image
        path = image_paths[iDrw] # ie 102_bath_copy.png 
        image_name = os.path.basename(path)

        if os.path.exists(path):
            img = Image.open(path).convert("RGB")
        else:
            print(f"Image not found: {path}")
            continue

        caption = captions.iloc[iDrw, 4:].values
        caption = ", ".join(str(x) for x in caption if pd.notna(x))
        full_caption = f"{caption}"

        image_embeds, text_embeds, joint_embedding = get_embedding(img, full_caption, model, processor, device)

        all_joined_embeddings.append(joint_embedding.cpu())
        all_text_embeddings.append(text_embeds.cpu())
        all_image_embeddings.append(image_embeds.cpu())
        filename_all.append(image_name)

    # save joined embeddings
    all_joined_embeddings = torch.cat(all_joined_embeddings).numpy()
    print(f"Joined embeddings have size: [{np.size(all_joined_embeddings[1,:])}, {np.size(all_joined_embeddings[:,1])}].")
    
    all_joined_embeddings = np.transpose(all_joined_embeddings)
    df = pd.DataFrame(all_joined_embeddings, index=None,
                    columns=[os.path.basename(p) for p in filename_all])
    outname = f"{embeddings_folder}{model}_joined.csv"
    df.to_csv(outname)
    print(f"Save {outname}")

    # save text embeddings
    all_text_embeddings = torch.cat(all_text_embeddings).numpy()
    print(f"Text embeddings have size: [{np.size(all_text_embeddings[1,:])}, {np.size(all_text_embeddings[:,1])}].")
    
    all_text_embeddings = np.transpose(all_text_embeddings)
    df = pd.DataFrame(all_text_embeddings, index=None,
                    columns=[os.path.basename(p) for p in filename_all])
    outname = f"{embeddings_folder}{model}_text.csv"
    df.to_csv(outname)
    print(f"Save {outname}")

    # save image embeddings
    all_image_embeddings = torch.cat(all_image_embeddings).numpy()
    print(f"Image embeddings have size: [{np.size(all_image_embeddings[1,:])}, {np.size(all_image_embeddings[:,1])}].")
    
    all_image_embeddings = np.transpose(all_image_embeddings)
    df = pd.DataFrame(all_image_embeddings, index=None,
                    columns=[os.path.basename(p) for p in filename_all])
    outname = f"{embeddings_folder}{model}_image.csv"
    df.to_csv(outname)
    print(f"Save {outname}") 


In [38]:
captions = load_captions(data_folder, num_repeats=4)
print(f"Found {len(captions)} captions.")
captions.head(10)

Found 544 captions.


,image,object_1,object_2,object_3,object_4,object_5,object_6,object_7,object_8,object_9,...,object_15,object_16,object_17,object_18,object_19,object_20,object_21,object_22,object_23,object_24
0,102_bath,door,towel holder,bathtub,sink,soap,cabinet,mirror,paper towel,toilet paper,...,shower screen,rug,toilet brush,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,102_bath,door,towel holder,bathtub,sink,soap,cabinet,mirror,paper towel,toilet paper,...,shower screen,rug,toilet brush,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,102_bath,door,towel holder,bathtub,sink,soap,cabinet,mirror,paper towel,toilet paper,...,shower screen,rug,toilet brush,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,102_bath,door,towel holder,bathtub,sink,soap,cabinet,mirror,paper towel,toilet paper,...,shower screen,rug,toilet brush,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,102_bath_copy,bathtub,shampoo bottle,shower,shower screen,bathroom vanity,sink,toothbrush,cup,soap,...,picture,window,plant,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,102_bath_copy,bathtub,shampoo bottle,shower,shower screen,bathroom vanity,sink,toothbrush,cup,soap,...,picture,window,plant,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,102_bath_copy,bathtub,shampoo bottle,shower,shower screen,bathroom vanity,sink,toothbrush,cup,soap,...,picture,window,plant,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,102_bath_copy,bathtub,shampoo bottle,shower,shower screen,bathroom vanity,sink,toothbrush,cup,soap,...,picture,window,plant,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,102_kitch,washing machine,fridge,cleaning utensil,cabinet,counter,plate,cup,stove,extractor hood,...,table,dish,chair,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,102_kitch,washing machine,fridge,cleaning utensil,cabinet,counter,plate,cup,stove,extractor hood,...,table,dish,chair,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
image_paths = load_images(data_folder)
print(f"Found {len(image_paths)} drawings.")
print(image_paths[:10])  # Display the first image paths for verification

Found 544 drawings.
['../../Drawings2/exp_gazeCon\\102_bath.png', '../../Drawings2/exp_gazeCon\\102_bath1_3D.png', '../../Drawings2/exp_gazeCon\\102_bath1_copy_3D.png', '../../Drawings2/exp_gazeCon\\102_bath2_3D.png', '../../Drawings2/exp_gazeCon\\102_bath2_copy_3D.png', '../../Drawings2/exp_gazeCon\\102_bath3_3D.png', '../../Drawings2/exp_gazeCon\\102_bath3_copy_3D.png', '../../Drawings2/exp_gazeCon\\102_bath_copy.png', '../../Drawings2/exp_gazeCon\\102_kitch.png', '../../Drawings2/exp_gazeCon\\102_kitch1_3D.png']


In [41]:
RDM_folder = embeddings_folder

embedding_types = ["joined", "text", "image"]
drawing_types = ["raw", "draw3D"]
conditions = ["typical", "copy"]
categories = ["kit", "bat"]

expected_n_subjects = 34

os.makedirs(RDM_folder, exist_ok=True)


# PARSE IMAGE NAME
def parse_image_name(name):

    # Remove path and extension
    stem = os.path.splitext(os.path.basename(name))[0]

    # Examples:
    # 102_bath
    # 102_bath1_3D
    # 102_bath1_copy_3D
    # 102_bath_copy
    # 102_kitch2_3D

    m = re.match(r"^(\d+)_([A-Za-z]+)", stem)

    if m is None:
        return None

    subject = m.group(1)
    category_raw = m.group(2).lower()

    # Standardize category names
    category_map = {
        "bath": "bat",
        "bat": "bat",
        "kitch": "kit",
        "kit": "kit",
    }

    category = category_map.get(category_raw, category_raw)

    drawing_type = "draw3D" if "_3d" in stem.lower() else "raw"
    condition = "copy" if "_copy" in stem.lower() else "typical"

    return {
        "image": name,
        "subject": subject,
        "category": category,
        "drawing_type": drawing_type,
        "condition": condition
    }


# LOOP OVER EMBEDDING TYPES
rdm_results = {}

for embedding_type in embedding_types:

    embedding_file = os.path.join(
        embeddings_folder,
        f"clip_{embedding_type}.csv"
    )

    print(f"\nLoading {embedding_file}")

    emd_df = pd.read_csv(embedding_file)

    # First column contains feature names; remaining columns = images
    image_names = emd_df.columns[1:].tolist()

    # rows = images, columns = embedding dimensions
    embeddings = emd_df.iloc[:, 1:].T
    embeddings.index = image_names

    # Parse all image names
    metadata = pd.DataFrame([
        parse_image_name(name)
        for name in image_names
        if parse_image_name(name) is not None
    ])

    metadata = metadata.set_index("image")


    # ALL CONDITION COMBINATIONS
    for drawing_type in drawing_types:
        for condition in conditions:
            for category in categories:

                subset = metadata[
                    (metadata["drawing_type"] == drawing_type) &
                    (metadata["condition"] == condition) &
                    (metadata["category"] == category)
                ].copy()

                selected = subset.index.tolist()

                if len(selected) == 0:
                    print(
                        f"Skipping {embedding_type} | "
                        f"{drawing_type} | {condition} | {category}"
                    )
                    continue

                # Subject order
                subjects = sorted(
                    subset["subject"].unique(),
                    key=int
                )

                # CHECK NUMBER OF SUBJECTS / IMAGES
                if len(subjects) != expected_n_subjects:
                    raise ValueError(
                        f"{embedding_type} | {drawing_type} | "
                        f"{condition} | {category}: "
                        f"found {len(subjects)} subjects instead of "
                        f"{expected_n_subjects}"
                    )

                expected_repeats = 3 if drawing_type == "draw3D" else 1

                counts = subset.groupby("subject").size()

                bad_counts = counts[counts != expected_repeats]

                if len(bad_counts):
                    raise ValueError(
                        f"{embedding_type} | {drawing_type} | "
                        f"{condition} | {category}: unexpected image counts:\n"
                        f"{bad_counts}"
                    )


                # IMAGE-LEVEL SPEARMAN CORRELATIONS
                X = embeddings.loc[selected].to_numpy(dtype=float)

                image_r = spearmanr(
                    X,
                    axis=1
                ).statistic

                image_r = np.asarray(image_r)

                # COLLAPSE IMAGE CORRELATIONS TO SUBJECT LEVEL
                subject_r = np.zeros(
                    (len(subjects), len(subjects))
                )

                subject_array = subset.loc[selected, "subject"].to_numpy()

                # Image indices belonging to each subject
                subject_indices = {
                    subject: np.where(subject_array == subject)[0]
                    for subject in subjects
                }

                for i, subject1 in enumerate(subjects):

                    # Self-similarity
                    subject_r[i, i] = 1

                    for j in range(i + 1, len(subjects)):

                        subject2 = subjects[j]

                        idx1 = subject_indices[subject1]
                        idx2 = subject_indices[subject2]

                        # raw:    1 x 1 correlation
                        # draw3D:  3 x 3 correlations
                        block = image_r[np.ix_(idx1, idx2)]

                        mean_r = np.nanmean(block)

                        subject_r[i, j] = mean_r
                        subject_r[j, i] = mean_r


                # RDM = 1 - SPEARMAN R
                rdm = 1 - subject_r
                np.fill_diagonal(rdm, 0)

                rdm_df = pd.DataFrame(
                    rdm,
                    index=subjects,
                    columns=subjects
                )

                # SAVE
                outfile = os.path.join(
                    RDM_folder,
                    f"rdm_clip_{drawing_type}_{embedding_type}_"
                    f"{category}_{condition}.csv"
                )

                rdm_df.to_csv(outfile)

                rdm_results[
                    (
                        embedding_type,
                        drawing_type,
                        condition,
                        category
                    )
                ] = rdm_df

                print(
                    f"Saved {rdm_df.shape}: "
                    f"{os.path.basename(outfile)}"
                )


Loading ../../dnn_features/clip/clip_joined.csv
Saved (34, 34): rdm_clip_raw_joined_kit_typical.csv
Saved (34, 34): rdm_clip_raw_joined_bat_typical.csv
Saved (34, 34): rdm_clip_raw_joined_kit_copy.csv
Saved (34, 34): rdm_clip_raw_joined_bat_copy.csv
Saved (34, 34): rdm_clip_draw3D_joined_kit_typical.csv
Saved (34, 34): rdm_clip_draw3D_joined_bat_typical.csv
Saved (34, 34): rdm_clip_draw3D_joined_kit_copy.csv
Saved (34, 34): rdm_clip_draw3D_joined_bat_copy.csv

Loading ../../dnn_features/clip/clip_text.csv
Saved (34, 34): rdm_clip_raw_text_kit_typical.csv
Saved (34, 34): rdm_clip_raw_text_bat_typical.csv
Saved (34, 34): rdm_clip_raw_text_kit_copy.csv
Saved (34, 34): rdm_clip_raw_text_bat_copy.csv
Saved (34, 34): rdm_clip_draw3D_text_kit_typical.csv
Saved (34, 34): rdm_clip_draw3D_text_bat_typical.csv
Saved (34, 34): rdm_clip_draw3D_text_kit_copy.csv
Saved (34, 34): rdm_clip_draw3D_text_bat_copy.csv

Loading ../../dnn_features/clip/clip_image.csv
Saved (34, 34): rdm_clip_raw_image_kit_t